## Import libraries


In [ ]:
import json
import os
from glob import glob

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import Point
from tqdm import tqdm

# --- Repo-relative paths ---------------------------------------------------
# Resolved from this notebook's location, so the checkout can be moved or cloned
# anywhere without editing paths here. Override with SOIL_SCENARIOS_ROOT.
import os
from pathlib import Path


def _find_repo_root(start=None):
    env = os.environ.get("SOIL_SCENARIOS_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path(start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "simplace").is_dir() and (cand / "orchestration").is_dir():
            return cand
    raise FileNotFoundError(
        f"repo root not found from {here} (looked for simplace/ + orchestration/)"
    )


REPO_ROOT = _find_repo_root()

MAIN_DATA_DIR = os.path.join(str(REPO_ROOT), "data")
# External (outside the repo): shared cluster data stores.
EXTERNAL_DATA_DIR = os.environ.get("EXTERNAL_DATA_DIR", "/beegfs/halder/DATA")
DWD_CLIMATE_DIR = os.environ.get(
    "DWD_CLIMATE_DIR",
    "/beegfs/common/data/climate/dwd/csvs/germany_ubn_1951-01-01_to_2024-08-30",
)
INTERIM_DATA_DIR = os.path.join(MAIN_DATA_DIR, "interim")
PROCESSED_DATA_DIR = os.path.join(MAIN_DATA_DIR, "processed")

## Read the datasets


In [ ]:
project_df = pd.read_csv(os.path.join(MAIN_DATA_DIR, "project.csv"))

# Read the soil coordinates
soil_coords = gpd.read_file(os.path.join(MAIN_DATA_DIR, "Site_Soil_BZE_WGS84.gpkg"))
soil_coords = soil_coords[["PointID", "NUTS_ID", "NUTS_NAME", "STATE_NAME", "geometry"]]

print(project_df.shape)
project_df.head()

## Prepare the project file for soil points


In [ ]:
crops = [
    "winter_wheat",
    "winter_rapeseed",
    "maize",
    "spring_barley",
    "sugar_beet",
    "potato",
]

harvest_next_year = ["winter_wheat", "winter_rapeseed"]

for crop in crops:
    phenology_df = pd.read_csv(
        os.path.join(PROCESSED_DATA_DIR, "phenology", "main", f"phenology_{crop}.csv")
    )

    sowing_data = phenology_df[["PointID", "sowing_date", "sowing_doy"]].copy()
    sowing_data["sowing_date"] = pd.to_datetime(
        sowing_data["sowing_date"], format="%Y-%m-%d"
    )
    sowing_data["year"] = sowing_data["sowing_date"].dt.year
    sowing_data = sowing_data[["PointID", "year", "sowing_date", "sowing_doy"]]

    sowing_data_transformed = sowing_data.copy()
    sowing_data_transformed.rename(
        columns={"PointID": "vPointID", "sowing_doy": "vIDPL"}, inplace=True
    )
    sowing_data_transformed["start_date"] = pd.to_datetime(
        sowing_data_transformed["year"].astype(str) + "-01-01"
    )

    if crop in harvest_next_year:
        sowing_data_transformed["end_date"] = pd.to_datetime(
            (sowing_data_transformed["year"] + 1).astype(str) + "-12-31"
        )
    else:
        sowing_data_transformed["end_date"] = pd.to_datetime(
            (sowing_data_transformed["year"]).astype(str) + "-12-31"
        )

    sowing_data_transformed.drop(columns=["year", "sowing_date"], inplace=True)

    # Prepare the project file
    project_df_final = project_df.copy()
    project_df_final.drop(
        columns=["vPrecipitat", "vTempMean", "vRadiation", "vcluster", "vlon", "vlat"],
        inplace=True,
    )
    project_df_final = pd.merge(
        left=project_df_final,
        right=sowing_data_transformed,
        on=["vPointID"],
        how="inner",
    )
    col_order = [
        "projectid",
        "simulationid",
        "vColumn",
        "vRow",
        "vPointID",
        "vNUTS_ID",
        "vNUTS_NAME",
        "vSTATE_NAME",
        "start_date",
        "end_date",
        "vIDPL",
    ]
    project_df_final = project_df_final[col_order]
    project_df_final["simulationid"] = np.arange(1, len(project_df_final) + 1)
    project_df_final["vYear"] = project_df_final["end_date"].dt.year
    project_df_final.rename(columns={"vPointID": "vLocationID"}, inplace=True)

    projectid = project_df_final["projectid"]
    simulationid = project_df_final["simulationid"]

    project_df_final["projectid"] = simulationid
    project_df_final["simulationid"] = projectid

    out_path = os.path.join(
        PROCESSED_DATA_DIR, "simplace", "main_project_file", f"project_{crop}.csv"
    )
    # project_df_final.to_csv(out_path, index=False, sep=";")

    print(
        f"Project file for {crop} has been saved at path: {out_path} | Data shape: {project_df_final.shape}"
    )

## Prepare the project files for crop specific LAI points


In [ ]:
# Load germany shapefile
de_nuts1_gdf = gpd.read_file(
    os.path.join("/beegfs", "halder", "DATA", "DE_NUTS", "DE_NUTS_3.shp")
)
de_nuts1_gdf = de_nuts1_gdf[
    de_nuts1_gdf["LEVL_CODE"] == 1
]  # filter only NUTS1 level code
de_nuts1_gdf = de_nuts1_gdf[["NUTS_NAME", "geometry"]]
de_nuts1_gdf.rename(columns={"NUTS_NAME": "STATE_NAME"}, inplace=True)
de_nuts1_gdf.to_crs(epsg=25832, inplace=True)

de_nuts3_gdf = gpd.read_file(
    os.path.join("/beegfs", "halder", "DATA", "DE_NUTS", "DE_NUTS_3.shp")
)
de_nuts3_gdf = de_nuts3_gdf[
    de_nuts3_gdf["LEVL_CODE"] == 3
]  # filter only NUTS3 level code
de_nuts3_gdf = de_nuts3_gdf[["NUTS_ID", "NUTS_NAME", "geometry"]]
de_nuts3_gdf.to_crs(epsg=25832, inplace=True)

print(de_nuts3_gdf.shape)
de_nuts3_gdf.head()

In [ ]:
crop_map_dict = {
    "Wheat": "winter_wheat",
    "Barley": "spring_barley",
    "Maize": "maize",
    "Potatoes": "potato",
    "Sugar Beet": "sugar_beet",
    "Rapeseed": "winter_rapeseed",
}

crop_samples_path = os.path.join(
    PROCESSED_DATA_DIR, "crop_type_samples", "crop_type_samples.gpkg"
)
crop_samples = gpd.read_file(crop_samples_path)
crop_samples["crop_type"] = crop_samples["crop_type"].replace(crop_map_dict)
crop_samples.to_crs(crs="EPSG:25832", inplace=True)

crop_samples = gpd.sjoin(
    left_df=crop_samples, right_df=de_nuts3_gdf, how="left", predicate="intersects"
)
crop_samples.drop("index_right", axis=1, inplace=True)
crop_samples = gpd.sjoin(
    left_df=crop_samples, right_df=de_nuts1_gdf, how="left", predicate="intersects"
)
crop_samples.drop("index_right", axis=1, inplace=True)
crop_samples.dropna(subset=["NUTS_ID"], inplace=True)

# Filter the data based on soil data availability
crop_samples_soil = pd.read_csv(
    os.path.join(PROCESSED_DATA_DIR, "soil", "lai", "LAI_soil.csv")
)
crop_samples = crop_samples[
    crop_samples["point_id"].isin(crop_samples_soil["point_id"])
]

In [ ]:
# Specify the DE DWD grip file path
DE_DWD_json_path = os.path.join(
    EXTERNAL_DATA_DIR, "DE_DWD_Lat_Lon", "latlon_to_rowcol.json"
)

with open(DE_DWD_json_path) as f:
    data = json.load(f)

# Convert data to GeoDataFrame
records = []
for coord, index in data:
    lat, lon = coord
    row, col = index
    point = Point(lon, lat)
    records.append({"row": row, "col": col, "geometry": point})

latlon_gdf = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
latlon_gdf = latlon_gdf.to_crs(f"EPSG:{25832}")

crop_samples = gpd.sjoin_nearest(left_df=crop_samples, right_df=latlon_gdf, how="left")

crop_samples.drop(columns="index_right", inplace=True)

In [ ]:
file_paths = glob(
    os.path.join(DWD_CLIMATE_DIR, "*", "*.csv.gz")
)
valid_flags = []

for i, row in tqdm(crop_samples.iterrows()):
    r = row["row"]
    c = row["col"]
    path = os.path.join(
            DWD_CLIMATE_DIR, str(r), f"daily_mean_RES1_C{c}R{r}.csv.gz"
        )

    if path in file_paths:
        valid_flags.append(True)
    else:
        valid_flags.append(False)

crop_samples["valid"] = valid_flags
crop_samples = crop_samples[crop_samples["valid"] == True]
print(crop_samples.shape)
crop_samples.head()

In [ ]:
harvest_next_year = ["winter_wheat", "winter_rapeseed"]

for crop in crops:
    phenology_df = pd.read_csv(
        os.path.join(
            PROCESSED_DATA_DIR, "phenology", "lai", f"phenology_{crop}_LAI.csv"
        )
    )
    phenology_df.rename(columns={"point_id": "PointID"}, inplace=True)

    sowing_data = phenology_df[["PointID", "sowing_date", "sowing_doy"]].copy()
    sowing_data["sowing_date"] = pd.to_datetime(
        sowing_data["sowing_date"], format="%Y-%m-%d"
    )
    sowing_data["year"] = sowing_data["sowing_date"].dt.year
    sowing_data = sowing_data[["PointID", "year", "sowing_date", "sowing_doy"]]

    sowing_data_transformed = sowing_data.copy()
    sowing_data_transformed.rename(
        columns={"PointID": "vPointID", "sowing_doy": "vIDPL"}, inplace=True
    )
    sowing_data_transformed["start_date"] = pd.to_datetime(
        sowing_data_transformed["year"].astype(str) + "-01-01"
    )

    if crop in harvest_next_year:
        sowing_data_transformed["end_date"] = pd.to_datetime(
            (sowing_data_transformed["year"] + 1).astype(str) + "-12-31"
        )
    else:
        sowing_data_transformed["end_date"] = pd.to_datetime(
            (sowing_data_transformed["year"]).astype(str) + "-12-31"
        )

    sowing_data_transformed.drop(columns=["year", "sowing_date"], inplace=True)
    sowing_data_transformed["vYear"] = sowing_data_transformed["end_date"].dt.year

    # Prepare the project file
    project_df_final = crop_samples.copy()
    project_df_final = project_df_final[project_df_final["crop_type"] == crop]
    project_df_final.rename(columns={"point_id": "vPointID"}, inplace=True)

    project_df_final = pd.merge(
        left=project_df_final,
        right=sowing_data_transformed,
        on=["vPointID"],
        how="inner",
    )

    project_df_final.rename(
        columns={
            "NUTS_ID": "vNUTS_ID",
            "NUTS_NAME": "vNUTS_NAME",
            "STATE_NAME": "vSTATE_NAME",
            "row": "vRow",
            "col": "vColumn",
            "vPointID": "vLocationID",
        },
        inplace=True,
    )

    project_df_final["simulationid"] = (
        "C"
        + project_df_final["vColumn"].astype(str)
        + "R"
        + project_df_final["vRow"].astype(str)
    )
    project_df_final.sort_values(by=["vLocationID", "year"], inplace=True)
    project_df_final["projectid"] = np.arange(1, len(project_df_final) + 1)

    col_order = [
        "projectid",
        "simulationid",
        "vColumn",
        "vRow",
        "vLocationID",
        "vNUTS_ID",
        "vNUTS_NAME",
        "vSTATE_NAME",
        "start_date",
        "end_date",
        "vIDPL",
        "vYear",
    ]
    project_df_final = project_df_final[col_order]

    out_path = os.path.join(
        PROCESSED_DATA_DIR, "simplace_project_file", "lai", f"project_{crop}.csv"
    )
    # project_df_final.to_csv(out_path, index=False, sep=";")

    print(
        f"Project file for {crop} has been saved at path: {out_path} | Data shape: {project_df_final.shape}"
    )